In [ ]:
from google import genai
from google.genai import types
from openai import OpenAI
import os
from dotenv import load_dotenv
from pypdf import PdfReader
import gradio as gr


In [ ]:
load_dotenv(override=True)
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

In [ ]:
reader = PdfReader("me/Profile.pdf")
linkedIn = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedIn += text

In [ ]:
print(linkedIn)

In [ ]:
with open ("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [ ]:
name = "Bhagirath Parmar"

In [ ]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedIn}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."

In [ ]:
system_prompt

In [ ]:
def chat(message, history):
    # Format history strictly for Gemini SDK
    formatted_history = []
    
    for m in history:
        role = "user" if m["role"] == "user" else "model"
        content = m["content"]
        
        if isinstance(content, str):
            text_value = content
        elif isinstance(content, list):
            # If it's a list, look for the first part that contains 'text'
            text_parts = [part.get("text", "") for part in content if isinstance(part, dict)]
            text_value = " ".join(text_parts)
        elif isinstance(content, dict):
            text_value = content.get("text", "")
        else:
            text_value = str(content)
        # -----------------------------

        formatted_history.append({
            "role": role,
            "parts": [{"text": text_value}]
        })
    chat_session = client.chats.create(
      model="gemini-2.5-flash",
      config=types.GenerateContentConfig(
        system_instruction=system_prompt,
        temperature=0.7,
      ),
      history=formatted_history
    )
    
    user_msg = message if isinstance(message, str) else message.get("text", "")
    
    response = chat_session.send_message(user_msg)
    
    return response.text

In [ ]:
gr.ChatInterface(
    fn=chat,  
    title=f"Chat with {name} (AI)"
).launch()

In [ ]:
from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str

In [ ]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedIn}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [ ]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [ ]:
openai = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [ ]:
def evaluate(reply, message, history) -> Evaluation:
    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = openai.chat.completions.parse(model="gpt-4o-mini", messages=messages, response_format=Evaluation)
    return response.choices[0].message.parsed

In [ ]:
user_query = "do you hold a patent?"
chat_session = client.chats.create(
      model="gemini-2.5-flash",
      config=types.GenerateContentConfig(
        system_instruction=system_prompt,
        temperature=0.7,
      )
)

response = chat_session.send_message(user_query)
reply = response.text

In [ ]:
reply

In [ ]:
evaluate(reply, user_query, [])

In [ ]:
def rerun(reply, message, history, feedback):
    # Construct the instruction for the retry
    updated_instruction = system_prompt + (
        f"\n\n## Previous answer rejected\n"
        f"Your previous attempt was rejected for quality. Please try again.\n"
        f"## Your attempted answer:\n{reply}\n\n"
        f"## Reason for rejection:\n{feedback}\n"
    )
    
    # We create a fresh chat for the rerun to ensure the new instruction is followed
    chat_session = client.chats.create(
        model="gemini-2.5-flash",
        config=types.GenerateContentConfig(
            system_instruction=updated_instruction,
            temperature=0.7,
        ),
        history=history # history must already be in Gemini format
    )
    
    response = chat_session.send_message(message)
    return response.text



In [ ]:
def chat(message, history):
    # 1. Determine the logic-specific system instruction
    if "patent" in message.lower():
        system = system_prompt + "\n\nEverything in your reply needs to be in pig latin - " \
                                "it is mandatory that you respond only and entirely in pig latin."
    else:
        system = system_prompt

    # 2. Format the Gradio history into Gemini's expected 'role'/'parts' format
    formatted_history = []
    for turn in history:
        role = "user" if turn["role"] == "user" else "model"
        raw_content = turn["content"]
        
        # Robustly extract the string text
        if isinstance(raw_content, str):
            text_value = raw_content
        elif isinstance(raw_content, list):
            # Gradio history items can be lists of dicts
            # We grab the 'text' key from the first part
            text_value = raw_content[0].get("text", "") if raw_content else ""
        elif isinstance(raw_content, dict):
            text_value = raw_content.get("text", "")
        else:
            text_value = str(raw_content)

        formatted_history.append({
            "role": role,
            "parts": [{"text": text_value}] # This must be a string!
        })

    # 3. Initialize the Gemini Chat Session
    chat_session = client.chats.create(
        model="gemini-2.5-flash",
        config=types.GenerateContentConfig(
            system_instruction=system,
            temperature=0.7,
        ),
        history=formatted_history
    )

    # 4. Get the initial reply
    response = chat_session.send_message(message)
    reply = response.text

    # 5. Evaluate (using your existing evaluation function)
    evaluation = evaluate(reply, message, history)
    
    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print(f"Failed evaluation - retrying. Feedback: {evaluation.feedback}")
        # Call rerun with the formatted history
        reply = rerun(reply, message, formatted_history, evaluation.feedback)
        
    return reply

In [ ]:
gr.ChatInterface(
    fn=chat,  
    title=f"Chat with {name} (AI)"
).launch()